# StrataForge Progress Notebook
## Phase 03 LLM Gateway
Purpose: exercise the Phase 03 gateway through deterministic noop, LiteLLM transport-compatible, OpenAI provider-native strict, and bounded repair examples.


### Environment Assumptions
- The deterministic notebook path does not require live provider calls.
- The mocked OpenAI path uses `PHASE03_NOTEBOOK_OPENAI_API_KEY` with a fake value.
- The optional live cell runs only when `STRATAFORGE_ENABLE_LIVE_LLM_NOTEBOOK=1` and `OPENAI_API_KEY` are both set.


In [ ]:
# environment setup
from pathlib import Path

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "phase03-progress"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"repo_root={REPO_ROOT}")
print(f"artifact_root={ARTIFACT_ROOT}")


In [ ]:
# imports
import json
import os

import httpx
from pydantic import BaseModel

from strataforge.domain import RepairKind, RepairRequest, TreeBuildRequest, TreeSettings
from strataforge.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayRepairEngine,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    LiteLLMProviderConfig,
    LiteLLMSDKAdapter,
    NoopProviderAdapter,
    NoopScriptedResponse,
    OpenAIProviderConfig,
    OpenAIResponsesHTTPAdapter,
    StructuredOutputMode,
)
from strataforge.tree import build_tree


In [ ]:
# configuration
class EchoResponse(BaseModel):
    message: str

os.environ.setdefault("PHASE03_NOTEBOOK_OPENAI_API_KEY", "mock-phase03-key")

noop_gateway = GatewayService(
    GatewayConfig(
        provider=LiteLLMProviderConfig(model="noop-model"),
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "noop-audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {"progress-noop": NoopScriptedResponse(output_json={"message": "hello from noop"})}
    ),
)


def fake_litellm_responses(**kwargs):
    return {
        "id": "litellm-progress-1",
        "output_text": '{"message":"hello from litellm"}',
        "usage": {"input_tokens": 5, "output_tokens": 6, "total_tokens": 11},
        "request_echo": kwargs["text"]["format"]["name"],
    }


litellm_gateway = GatewayService(
    GatewayConfig(
        provider=LiteLLMProviderConfig(model="openai/gpt-4.1-mini"),
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "litellm-audit")),
        structured_output_mode_preference=StructuredOutputMode.TRANSPORT_COMPATIBLE,
    ),
    provider_adapter=LiteLLMSDKAdapter(responses_callable=fake_litellm_responses),
)


def openai_handler(request: httpx.Request) -> httpx.Response:
    return httpx.Response(
        200,
        headers={"x-request-id": "phase03-progress-openai-request"},
        json={
            "id": "phase03-progress-openai-response",
            "output": [
                {
                    "content": [
                        {
                            "type": "output_text",
                            "text": '{"message":"hello from openai strict"}',
                        }
                    ]
                }
            ],
            "usage": {"input_tokens": 7, "output_tokens": 8, "total_tokens": 15},
        },
    )


openai_gateway = GatewayService(
    GatewayConfig(
        provider=OpenAIProviderConfig(
            model="gpt-4.1-mini",
            api_key_env_var="PHASE03_NOTEBOOK_OPENAI_API_KEY",
        ),
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "openai-audit")),
        structured_output_mode_preference=StructuredOutputMode.PROVIDER_NATIVE,
    ),
    provider_adapter=OpenAIResponsesHTTPAdapter(
        client=httpx.Client(
            transport=httpx.MockTransport(openai_handler),
            base_url="https://api.openai.com",
        )
    ),
)


def openai_repair_handler(request: httpx.Request) -> httpx.Response:
    payload = json.loads(request.content.decode("utf-8"))
    request_id = request.headers.get("Idempotency-Key", "repair-progress")
    if payload["text"]["format"]["name"] != "RepairPromptResponse":
        raise AssertionError("unexpected repair schema name")
    return httpx.Response(
        200,
        headers={"x-request-id": "phase03-progress-repair-request"},
        json={
            "id": "phase03-progress-repair-response",
            "output": [
                {
                    "content": [
                        {
                            "type": "output_text",
                            "text": json.dumps(
                                {
                                    "request_id": request_id,
                                    "status": "proposal_generated",
                                    "message": "normalize title casing",
                                    "proposed_title": "Overview",
                                }
                            ),
                        }
                    ]
                }
            ],
            "usage": {"input_tokens": 9, "output_tokens": 7, "total_tokens": 16},
        },
    )


repair_engine = GatewayRepairEngine(
    config=GatewayConfig(
        provider=OpenAIProviderConfig(
            model="gpt-4.1-mini",
            api_key_env_var="PHASE03_NOTEBOOK_OPENAI_API_KEY",
        ),
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "repair-audit")),
        structured_output_mode_preference=StructuredOutputMode.PROVIDER_NATIVE,
    ),
    provider_adapter=OpenAIResponsesHTTPAdapter(
        client=httpx.Client(
            transport=httpx.MockTransport(openai_repair_handler),
            base_url="https://api.openai.com",
        )
    ),
)


In [ ]:
# execution
noop_success = noop_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="progress-noop",
        messages=(LLMMessage(role=LLMRole.USER, content="return a greeting"),),
        response_model=EchoResponse,
        idempotency_key="progress-noop",
    )
)

litellm_success = litellm_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="progress-litellm",
        messages=(LLMMessage(role=LLMRole.USER, content="return a greeting"),),
        response_model=EchoResponse,
        idempotency_key="progress-litellm",
        structured_output_mode=StructuredOutputMode.TRANSPORT_COMPATIBLE,
    )
)

openai_success = openai_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="progress-openai",
        messages=(LLMMessage(role=LLMRole.USER, content="return a greeting"),),
        response_model=EchoResponse,
        idempotency_key="progress-openai",
        structured_output_mode=StructuredOutputMode.PROVIDER_NATIVE,
    )
)

repair_decisions = repair_engine.evaluate(
    (
        RepairRequest(
            request_id="progress-repair",
            subject_id="node-001",
            repair_kind=RepairKind.TITLE_NORMALIZATION,
            rationale="manual Phase 03 notebook exercise",
            details={"candidate_title": "overview"},
        ),
    )
)


In [ ]:
# execution
live_result = "skipped: set STRATAFORGE_ENABLE_LIVE_LLM_NOTEBOOK=1 and OPENAI_API_KEY to run live examples"
if os.getenv("STRATAFORGE_ENABLE_LIVE_LLM_NOTEBOOK") == "1" and os.getenv("OPENAI_API_KEY"):
    live_result = "live execution requested by environment; add a real provider request here during operator verification"
print(live_result)

tree_manifest = build_tree(
    TreeBuildRequest(
        parse_manifest_path=str(REPO_ROOT / "fixtures" / "phase02" / "inputs" / "clean_outline" / "manifest.json"),
        tree_run_id="progress-major-changes",
        settings=TreeSettings(max_pages_per_leaf_node=2),
    )
)
tree_strategy_report = json.loads(Path(tree_manifest.strategy_execution_report_path).read_text(encoding="utf-8"))
tree_decomposition_report = json.loads(Path(tree_manifest.decomposition_report_path).read_text(encoding="utf-8"))


In [ ]:
# inspect results
summary = {
    "noop": {
        "assurance": noop_success.assurance_mode.value,
        "message": noop_success.output.message,
        "audit_path": noop_success.audit_path,
    },
    "litellm": {
        "assurance": litellm_success.assurance_mode.value,
        "message": litellm_success.output.message,
        "attempts": len(litellm_success.attempts),
        "audit_path": litellm_success.audit_path,
    },
    "openai": {
        "assurance": openai_success.assurance_mode.value,
        "message": openai_success.output.message,
        "provider_request_id": openai_success.provider_request_id,
        "audit_path": openai_success.audit_path,
    },
    "repair": {
        "status": repair_decisions[0].status.value,
        "request_id": repair_decisions[0].request_id,
        "message": repair_decisions[0].message,
    },
    "tree_major_changes": {
        "selected_strategy": tree_strategy_report["selected_strategy"],
        "decomposition_method": tree_decomposition_report["decomposition_method"],
        "committed_node_count": tree_manifest.committed_node_count,
    },
    "live": live_result,
}
print(json.dumps(summary, indent=2, sort_keys=True))


### Known Limitations
- The LiteLLM notebook path demonstrates transport-compatible assurance only.
- The OpenAI notebook path is mocked by default; provider-native strictness is exercised against a mocked Responses payload, not a live network call.
- The repair notebook example uses the standalone `GatewayRepairEngine`, not a full tree build, to keep the canonical notebook deterministic and fast.
